In [38]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
from dotenv import load_dotenv
from pathlib import Path
load_dotenv(Path.cwd().parents[1] / ".env")  # если cwd = notebooks/week1




True

In [8]:
qdrant_client = QdrantClient(url="http://localhost:6333")

all_points = qdrant_client.scroll(
    collection_name="Amazon-item-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [10]:
all_points[0][0].payload


{'description': 'KEEPRO Pencil 2nd Generation for iPad, Magnetic Wireless Charge Tilt Sensitivity Palm Rejection Active Pen for Apple iPad Pro 11" 4/3/2/1, iPad Pro 12.9" 6/5/4/3, iPad Air 4/5, iPad Mini 6 [Compatibility]- ONLY compatible with iPad mini (6th generation), iPad Air (4th and 5th generation), iPad Pro 12.9-inch (3rd, 4th, 5th and 6th generation), iPad Pro 11-inch (1st, 2nd, 3rd and 4th generation), check and confirm your device before place the order (Note: If the pen doesn\'t charge, fully charge your iPad first then try charging the pen again) [Charging and Pairs Magnetically]- Charges wirelessly, attaches and pairs magnetically to the compatible iPad, this pen is a preferable alternative to the Apple Pencil 2nd Generation [Tilt Sensitivity & Pixel Precision]- Pixel-perfect precision and industry-leading low latency with tilt sensitivity making drawing, sketching, coloring, taking notes, and marking up PDFs, as easy and natural as a real pencil [Native Palm Rejection]- R

In [11]:
all_context = [{"id":data.payload["parent_asin"], "text":data.payload["description"]} for data in all_points[0]]

In [12]:
all_context

[{'id': 'B0BF18F6R7',
  'text': 'KEEPRO Pencil 2nd Generation for iPad, Magnetic Wireless Charge Tilt Sensitivity Palm Rejection Active Pen for Apple iPad Pro 11" 4/3/2/1, iPad Pro 12.9" 6/5/4/3, iPad Air 4/5, iPad Mini 6 [Compatibility]- ONLY compatible with iPad mini (6th generation), iPad Air (4th and 5th generation), iPad Pro 12.9-inch (3rd, 4th, 5th and 6th generation), iPad Pro 11-inch (1st, 2nd, 3rd and 4th generation), check and confirm your device before place the order (Note: If the pen doesn\'t charge, fully charge your iPad first then try charging the pen again) [Charging and Pairs Magnetically]- Charges wirelessly, attaches and pairs magnetically to the compatible iPad, this pen is a preferable alternative to the Apple Pencil 2nd Generation [Tilt Sensitivity & Pixel Precision]- Pixel-perfect precision and industry-leading low latency with tilt sensitivity making drawing, sketching, coloring, taking notes, and marking up PDFs, as easy and natural as a real pencil [Native Pa

In [16]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "Suggested question"},
            "chunk_ids": {"type": "array", "items": {"type": "string", "description":"ID of the chunk that could be used to answer the question"}},
            "answer_example":{"type": "string", "description": "Suggested answer grouped in the context"},
            "reasoning":{"type": "string", "description": "Reasoning why the question could be answered with the chunks"},
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{all_context}
"""

In [18]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[{'id': 'B0BF18F6R7', 'text': 'KEEPRO Pencil 2nd Generation for iPad, Magnetic Wireless Charge Tilt Sensitivity Palm Rejection Active Pen for Apple iPad Pro 11" 4/3/2/1, iPad Pro 12.9" 6/5/4/3, iPad Air 4/5, iPad Mini 6 [Compatibility]- ONLY compatible with iPad mini (6th generation), iPad Air (4th and 5th generation), iPad Pro 12.9-inch (3rd, 4th, 5th and 6th generation), iPad Pro 11-inch (1st, 2nd, 3rd and 4th generation), check and confirm your device before place the order (Note: If the pen doesn\'t charge, fully charge your iPad first then try charging the pen again) [Charging and Pairs Magnetically]- Charges wirelessly, attaches and pairs magnetically to the compatible iPad, this pen is a preferable alternative to the Apple Pencil 2nd Generation [Tilt Sensitivity & Pixel Precision]- Pixel-perfect precision and industry-leading low latency with tilt sensitivity making drawing, sketching, coloring, tak

In [25]:
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user","content": USER_PROMPT}
    ],
    reasoning_effort="minimal"
)

print(response.choices[0].message.content)

[
  {
    "question": "Is the KEEPRO Pencil compatible with my iPad Air 5?",
    "chunk_ids": [
      "B0BF18F6R7"
    ],
    "answer_example": "Yes. The KEEPRO Pencil 2nd Generation is compatible with iPad Air (4th and 5th generation), so it will work with an iPad Air 5.",
    "reasoning": "The chunk B0BF18F6R7 contains the product compatibility list and explicitly lists iPad Air 4th and 5th generations."
  },
  {
    "question": "What devices can I connect with the HOSONGIN TRS-to-dual-TS splitter?",
    "chunk_ids": [
      "B0B96LV4C5"
    ],
    "answer_example": "You can use the HOSONGIN 1/4\" TRS female to dual 1/4\" TS mono Y-splitter to connect a mixer, amplifier, headphones, speaker, or other devices with a 6.35mm TRS output to hi-fi stereo or pro audio gear with dual 6.35mm TS mono inputs.",
    "reasoning": "Chunk B0B96LV4C5 describes standard connectors and example use cases such as mixers, amplifiers and speakers."
  },
  {
    "question": "How much area can the Tenda A33

In [26]:
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [28]:
len(json_output)

60

In [29]:
points = qdrant_client.scroll(
    collection_name="Amazon-item-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B0B96LV4C5")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [30]:
points[0].payload

{'description': 'HOSONGIN 1/4 TRS Stereo Jack to Dual 1/4 TS Mono Y-Splitter Insert Cable, Nylon Braided Jacket Gold-Plated Plug Double Shielding Cable, 1 Feet EASILY CONNECT DEVICES: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch Y-Splitter Cable uses a 1/4" TRS stereo cable to connects a Mixer, Amplifier, Headphones, Speaker, or other devices with 6.35mm 1/4" TRS output to Hi-Fi stereo, speakers, or other pro audio gear with Dual 2x 6.35mm 1/4" TS Mono jack Stereo inputs. STANDARD CONNECTORS: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch TS Mono Y-Splitter Cable uses standard audio connectors, a 6.35mm(1/4 inch) TRS Female connector on one end, and two 1/4" TS Mono connectors on the other end. It splits a stereo signal into L-R audio over two 1/4 inch TS cables. CLEAN & CLEAR SOUND: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch TS Mono plug Y-Splitter Cable uses double shielded layers, tinned copper wire core, and corrosion-resistant gold-plated connector prevents

In [32]:
def get_description(parent_asin:str)->str:
    points = qdrant_client.scroll(
        collection_name="Amazon-item-collection-00",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]

In [33]:
get_description("B0B96LV4C5")

'HOSONGIN 1/4 TRS Stereo Jack to Dual 1/4 TS Mono Y-Splitter Insert Cable, Nylon Braided Jacket Gold-Plated Plug Double Shielding Cable, 1 Feet EASILY CONNECT DEVICES: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch Y-Splitter Cable uses a 1/4" TRS stereo cable to connects a Mixer, Amplifier, Headphones, Speaker, or other devices with 6.35mm 1/4" TRS output to Hi-Fi stereo, speakers, or other pro audio gear with Dual 2x 6.35mm 1/4" TS Mono jack Stereo inputs. STANDARD CONNECTORS: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch TS Mono Y-Splitter Cable uses standard audio connectors, a 6.35mm(1/4 inch) TRS Female connector on one end, and two 1/4" TS Mono connectors on the other end. It splits a stereo signal into L-R audio over two 1/4 inch TS cables. CLEAN & CLEAR SOUND: HOSONGIN 1/4 inch TRS Stereo Female to Dual 1/4 inch TS Mono plug Y-Splitter Cable uses double shielded layers, tinned copper wire core, and corrosion-resistant gold-plated connector prevents EMI & RFI Signa

In [34]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [35]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline",
)

In [37]:
for item in json_output:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_description": [get_description(id) for id in item["chunk_ids"]],
        }
    )